# Capstone Project I: Mutual Fund Analytics
## Day 4: Fund Performance Analytics & Scorecard Calculation

This notebook implements the quantitative analysis of 40 mutual fund schemes over a 4.4-year period (from `2022-01-03` to `2026-05-29`).

### Tasks Performed:
1. **Compute Daily Returns** and validate distribution.
2. **Compute CAGR** for 1-Year, 3-Year, and 5-Year (Max History).
3. **Compute Sharpe and Sortino Ratios** ($R_f = 6.5\%$) to rank all 40 funds.
4. **Compute Alpha and Beta** via OLS regression against the Nifty 100 benchmark.
5. **Compute Maximum Drawdown** and identify peak-to-trough worst date ranges.
6. **Build a Composite Fund Scorecard (0-100)** to rank funds using a weighted rank-based system.
7. **Plot Benchmark Comparison Chart** for the top 5 funds vs Nifty 50 and Nifty 100 over a 3-year period and calculate tracking errors.

### 1. Setup & Data Loading

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
import scipy.stats
import matplotlib.pyplot as plt
import seaborn as sns

# Paths
base_dir = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
db_path = os.path.join(base_dir, 'data', 'db', 'bluestock_mf.db')
bench_path = os.path.join(base_dir, 'data', 'processed', '10_benchmark_indices_clean.csv')

conn = sqlite3.connect(db_path)

# Load Fund Info and NAV History
df_funds = pd.read_sql_query("SELECT amfi_code, scheme_name, expense_ratio_pct, benchmark, category FROM dim_fund", conn)
df_nav = pd.read_sql_query("SELECT amfi_code, date, nav FROM fact_nav ORDER BY amfi_code, date", conn)
df_nav['date'] = pd.to_datetime(df_nav['date'])

df_bench = pd.read_csv(bench_path)
df_bench['date'] = pd.to_datetime(df_bench['date'])

print(f"Loaded {len(df_funds)} funds, {len(df_nav)} NAV rows, and {len(df_bench)} benchmark rows.")

### 2. Compute Daily Returns & Validate Distribution

In [ ]:
# Compute daily return for all funds
df_nav['daily_return'] = df_nav.groupby('amfi_code')['nav'].pct_change().fillna(0.0)

# Summary statistics of returns across all funds
returns_summary = df_nav.groupby('amfi_code')['daily_return'].describe()
print("Daily Returns Summary Statistics (First 5 Funds):")
display(returns_summary.head())

# Plot Return Distributions (Combined Histogram)
plt.figure(figsize=(10, 6))
sns.histplot(df_nav['daily_return'], bins=100, kde=True, color='#1f77b4')
plt.title("Distribution of Daily Returns Across All 40 Funds (2022-2026)", fontsize=12, fontweight='bold')
plt.xlabel("Daily Return")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

### 3. Compute CAGR (1-Year, 3-Year, 5-Year Proxy)

In [ ]:
# Setup periods
end_date = pd.to_datetime('2026-05-29')
start_1yr = pd.to_datetime('2025-05-29')
start_3yr = pd.to_datetime('2023-05-29')
start_5yr = pd.to_datetime('2022-01-03') # Max available history

n_years_5yr = (end_date - start_5yr).days / 365.25
print(f"Max available history period: {n_years_5yr:.4f} years.")

cagr_list = []

for code, group in df_nav.groupby('amfi_code'):
    group = group.sort_values('date')
    
    nav_end = group[group['date'] == end_date]['nav'].values[0]
    nav_1yr = group[group['date'] == start_1yr]['nav'].values[0]
    nav_3yr = group[group['date'] == start_3yr]['nav'].values[0]
    nav_5yr = group[group['date'] == start_5yr]['nav'].values[0]
    
    cagr_1 = (nav_end / nav_1yr) ** (1 / 1.0) - 1
    cagr_3 = (nav_end / nav_3yr) ** (1 / 3.0) - 1
    cagr_5 = (nav_end / nav_5yr) ** (1 / n_years_5yr) - 1
    
    cagr_list.append({
        'amfi_code': code,
        'cagr_1yr_pct': cagr_1 * 100,
        'cagr_3yr_pct': cagr_3 * 100,
        'cagr_5yr_pct': cagr_5 * 100
    })

df_cagr = pd.DataFrame(cagr_list)
df_cagr_comp = pd.merge(df_funds[['amfi_code', 'scheme_name']], df_cagr, on='amfi_code')

print("CAGR Comparison Table (First 10 Funds):")
display(df_cagr_comp.head(10))

### 4. Compute Sharpe & Sortino Ratios

In [ ]:
ratios_list = []
Rf_daily = 0.065 / 252 # Annual risk-free rate of 6.5%

for code, group in df_nav.groupby('amfi_code'):
    daily_ret = group.sort_values('date')['daily_return'].values[1:] # Drop first day return (NaN -> 0)
    
    mean_daily = np.mean(daily_ret)
    std_daily = np.std(daily_ret, ddof=1)
    
    # Sharpe Ratio
    mean_ret_ann = mean_daily * 252
    vol_ann = std_daily * np.sqrt(252)
    sharpe = (mean_ret_ann - 0.065) / vol_ann if vol_ann > 0 else 0.0
    
    # Sortino Ratio (negative returns standard deviation)
    downside_ret = daily_ret[daily_ret < 0]
    downside_std_daily = np.std(downside_ret, ddof=1) if len(downside_ret) > 1 else 0.0
    downside_vol_ann = downside_std_daily * np.sqrt(252)
    sortino = (mean_ret_ann - 0.065) / downside_vol_ann if downside_vol_ann > 0 else 0.0
    
    ratios_list.append({
        'amfi_code': code,
        'volatility_ann_pct': vol_ann * 100,
        'sharpe': sharpe,
        'sortino': sortino
    })
    
df_ratios = pd.DataFrame(ratios_list)
df_ratios_comp = pd.merge(df_funds[['amfi_code', 'scheme_name']], df_ratios, on='amfi_code')

print("Sharpe & Sortino Ratios (Top 10 sorted by Sharpe):")
display(df_ratios_comp.sort_values('sharpe', ascending=False).head(10))

### 5. Compute Alpha & Beta (OLS Regression on Nifty 100)

In [ ]:
# Calculate Nifty 100 daily returns
df_bench['daily_return'] = df_bench.groupby('index_name')['close_value'].pct_change().fillna(0.0)
df_bench_ret_pivot = df_bench.pivot(index='date', columns='index_name', values='daily_return')

regression_list = []

for code, group in df_nav.groupby('amfi_code'):
    fund_ret = group.sort_values('date')['daily_return'].values[1:]
    fund_dates = group.sort_values('date')['date'].values[1:]
    
    # Get Nifty 100 returns aligned to fund dates
    nifty_ret = df_bench_ret_pivot['NIFTY100'].loc[fund_dates].values
    
    # Run OLS Regression
    slope, intercept, r_val, p_val, std_err = scipy.stats.linregress(nifty_ret, fund_ret)
    beta = slope
    alpha = intercept * 252 # Annualized alpha
    
    regression_list.append({
        'amfi_code': code,
        'alpha_pct': alpha * 100,
        'beta': beta
    })
    
df_regression = pd.DataFrame(regression_list)
df_reg_comp = pd.merge(df_funds[['amfi_code', 'scheme_name']], df_regression, on='amfi_code')

print("Alpha and Beta Metrics (Top 10 sorted by Alpha):")
display(df_reg_comp.sort_values('alpha_pct', ascending=False).head(10))

### 6. Compute Maximum Drawdown & Date Range

In [ ]:
drawdown_list = []

for code, group in df_nav.groupby('amfi_code'):
    group = group.sort_values('date').copy()
    group['running_max'] = group['nav'].cummax()
    group['drawdown'] = group['nav'] / group['running_max'] - 1
    
    max_dd = group['drawdown'].min()
    trough_idx = group['drawdown'].idxmin()
    trough_row = group.loc[trough_idx]
    trough_date = trough_row['date']
    
    # Preceding Peak Date
    preceding = group[group['date'] <= trough_date]
    peak_idx = preceding['nav'].idxmax()
    peak_row = group.loc[peak_idx]
    peak_date = peak_row['date']
    peak_nav = peak_row['nav']
    
    # Recovery Date
    post_trough = group[group['date'] > trough_date]
    recovery = post_trough[post_trough['nav'] >= peak_nav]
    if len(recovery) > 0:
        rec_date = recovery.iloc[0]['date'].strftime('%Y-%m-%d')
    else:
        rec_date = "Not Recovered"
        
    drawdown_list.append({
        'amfi_code': code,
        'max_dd_pct': max_dd * 100,
        'peak_date': peak_date.strftime('%Y-%m-%d'),
        'trough_date': trough_date.strftime('%Y-%m-%d'),
        'recovery_date': rec_date
    })
    
df_drawdown = pd.DataFrame(drawdown_list)
df_dd_comp = pd.merge(df_funds[['amfi_code', 'scheme_name']], df_drawdown, on='amfi_code')

print("Maximum Drawdown Summary (Top 10 sorted by worst drawdown, i.e., lowest DD):")
display(df_dd_comp.sort_values('max_dd_pct', ascending=True).head(10))

### 7. Construct Composite Fund Scorecard

In [ ]:
# Combine all computed metrics
df_metrics = df_funds[['amfi_code', 'scheme_name', 'category', 'expense_ratio_pct', 'benchmark']].copy()
df_metrics = df_metrics.merge(df_cagr, on='amfi_code')
df_metrics = df_metrics.merge(df_ratios, on='amfi_code')
df_metrics = df_metrics.merge(df_regression, on='amfi_code')
df_metrics = df_metrics.merge(df_drawdown, on='amfi_code')

# Compute Ranks (scaled 0 to 100)
df_metrics['rank_3yr'] = df_metrics['cagr_3yr_pct'].rank(pct=True) * 100
df_metrics['rank_sharpe'] = df_metrics['sharpe'].rank(pct=True) * 100
df_metrics['rank_alpha'] = df_metrics['alpha_pct'].rank(pct=True) * 100
df_metrics['rank_expense'] = df_metrics['expense_ratio_pct'].rank(ascending=False, pct=True) * 100 # Lower is better
df_metrics['rank_max_dd'] = df_metrics['max_dd_pct'].rank(ascending=True, pct=True) * 100 # Less negative is better (ascending=True is correct)

# Compute Composite Score
df_metrics['composite_score'] = (
    0.30 * df_metrics['rank_3yr'] +
    0.25 * df_metrics['rank_sharpe'] +
    0.20 * df_metrics['rank_alpha'] +
    0.15 * df_metrics['rank_expense'] +
    0.10 * df_metrics['rank_max_dd']
)

# Sort by score
df_metrics = df_metrics.sort_values('composite_score', ascending=False).reset_index(drop=True)
df_metrics['rank'] = df_metrics.index + 1

print("Top 10 Funds by Composite Performance Scorecard:")
display(df_metrics[['rank', 'amfi_code', 'scheme_name', 'category', 'cagr_3yr_pct', 'sharpe', 'alpha_pct', 'max_dd_pct', 'composite_score']].head(10))
# Save outputs
df_scorecard = df_metrics[[
    'rank', 'amfi_code', 'scheme_name', 'category', 'expense_ratio_pct',
    'cagr_3yr_pct', 'sharpe', 'alpha_pct', 'max_dd_pct', 'composite_score'
]].copy()
df_scorecard.to_csv(os.path.join(base_dir, 'fund_scorecard.csv'), index=False)
df_scorecard.to_csv(os.path.join(base_dir, 'reports', 'fund_scorecard.csv'), index=False)
df_alpha_beta = df_metrics[['amfi_code', 'scheme_name', 'alpha_pct', 'beta', 'volatility_ann_pct']]
df_alpha_beta.to_csv(os.path.join(base_dir, 'alpha_beta.csv'), index=False)
df_alpha_beta.to_csv(os.path.join(base_dir, 'reports', 'alpha_beta.csv'), index=False)
print("Saved fund_scorecard.csv and alpha_beta.csv to root and reports directories.")

### 8. Plot Benchmark Comparison Chart & Calculate Tracking Error

In [ ]:
# Setup mapping and pivot close prices
benchmark_mapping = {
    'NIFTY 100 TRI': 'NIFTY100',
    'BSE 250 SmallCap TRI': 'BSE_SMALLCAP',
    'CRISIL Dynamic Gilt Index': 'CRISIL_GILT',
    'NIFTY Midcap 150 TRI': 'NIFTY_MIDCAP150',
    'CRISIL Short Term Bond Index': 'CRISIL_LIQUID',
    'NIFTY 500 TRI': 'NIFTY500',
    'CRISIL Liquid Fund AI Index': 'CRISIL_LIQUID',
    'NIFTY 50 TRI': 'NIFTY50',
    'NIFTY Midcap 50 TRI': 'NIFTY_MIDCAP150',
    'NIFTY Large Midcap 250 TRI': 'NIFTY500'
}
df_bench_pivot = df_bench.pivot(index='date', columns='index_name', values='close_value')
df_bench_ret_pivot = df_bench.pivot(index='date', columns='index_name', values='daily_return')

# Get top 5 funds
top_5 = df_metrics.head(5)
dates_3yr = pd.date_range(start='2023-05-29', end='2026-05-29', freq='D')

plt.figure(figsize=(12, 7))

# Plot Benchmarks
nifty50_prices = df_bench_pivot['NIFTY50'].loc[dates_3yr]
nifty50_cum = (nifty50_prices / nifty50_prices.iloc[0]) * 100
plt.plot(nifty50_cum.index, nifty50_cum.values, label='Nifty 50 TRI (Benchmark)', color='#555555', linewidth=2.0, linestyle='--')

nifty100_prices = df_bench_pivot['NIFTY100'].loc[dates_3yr]
nifty100_cum = (nifty100_prices / nifty100_prices.iloc[0]) * 100
plt.plot(nifty100_cum.index, nifty100_cum.values, label='Nifty 100 TRI (Benchmark)', color='#888888', linewidth=2.0, linestyle='-.')

# Plot Top 5 funds
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
tracking_errors = []

for idx, fund_row in top_5.iterrows():
    code = fund_row['amfi_code']
    name = fund_row['scheme_name']
    short_name = name.split(" - ")[0]
    
    fund_nav = df_nav[(df_nav['amfi_code'] == code) & (df_nav['date'].isin(dates_3yr))].sort_values('date')
    
    nav_start = fund_nav['nav'].iloc[0]
    fund_cum = (fund_nav['nav'] / nav_start) * 100
    plt.plot(fund_nav['date'], fund_cum, label=f"{short_name} (Rank {fund_row['rank']})", color=colors[idx], linewidth=1.8)
    
    # Compute Tracking Error
    bench_col = benchmark_mapping.get(fund_row['benchmark'], 'NIFTY100')
    fund_ret = fund_nav['daily_return'].values[1:]
    bench_ret = df_bench_ret_pivot[bench_col].loc[fund_nav['date']].values[1:]
    
    te = np.std(fund_ret - bench_ret, ddof=1) * np.sqrt(252)
    tracking_errors.append({
        'scheme_name': name,
        'benchmark': fund_row['benchmark'],
        'tracking_error_pct': te * 100
    })

plt.title("3-Year Cumulative Performance Comparison: Top 5 Funds vs Benchmarks (2023-2026)", fontsize=13, fontweight='bold', pad=15)
plt.xlabel("Date", fontsize=11, labelpad=10)
plt.ylabel("Normalized Value (Base = 100 on 2023-05-29)", fontsize=11, labelpad=10)
plt.legend(loc='upper left', frameon=True, facecolor='#ffffff', edgecolor='#dddddd')
plt.tight_layout()
plt.savefig(os.path.join(base_dir, 'benchmark_comparison_chart.png'), dpi=300)
plt.show()

print("Tracking Errors for Top 5 Funds:")
display(pd.DataFrame(tracking_errors))